# 🌊 Flood Detection from Images using Deep Learning

Fine-tunes Google's lightweight **MobileNet** CNN (pre-trained on ImageNet) into a binary
**Flooding vs. No Flooding** image classifier using transfer learning with Keras / TensorFlow.

| | |
|---|---|
| **Base model** | MobileNet (ImageNet weights, `include_top=False`) |
| **Input size** | 224 x 224 x 3 |
| **Head** | GlobalAveragePooling2D → Dropout(0.2) → Dense(2, softmax) |
| **Loss / Optimizer** | Categorical cross-entropy / Adam (lr = 1e-4) |
| **Trainable** | Last 23 layers (rest frozen) |

> **▶ Prefer zero setup?** Use the companion notebook **`Flood_Detection_Colab.ipynb`** —
> it downloads the data & pre-trained weights automatically and lets you upload your own
> (satellite / drone / street-level) images straight from the browser.

**Data & pre-trained model (Google Drive)**
- Dataset (train / valid / test): https://drive.google.com/file/d/1oT07-wVuYj_hByn4FqUtIj63Yg_fe7dP/view
- Pre-trained model: https://drive.google.com/file/d/1mF8NmMClUbKXYJoDdW3OYHhwiSknI5hk/view

*"Flooding" images were collected by the authors of the paper
["Detecting floodwater on roadways from image data with handcrafted features and deep transfer learning"](https://arxiv.org/pdf/1909.00125.pdf);
"No Flooding" images were collected via an automated Google image search (may contain some label noise).*

## 1️⃣ Setup

Imports, reproducibility seeds and configuration. If the `data/` folder is not present,
the dataset zip (~52 MB) is fetched automatically from Google Drive.

In [ ]:
# Imports & reproducibility ---------------------------------------------------
import os, shutil, zipfile, random, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

LABELS = ['Flooding', 'No Flooding']   # index 0 / 1 -> alphabetical: flooding, normal
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

print('TensorFlow version :', tf.__version__)
print('GPUs available     :', tf.config.list_physical_devices('GPU'))

In [ ]:
# Get the dataset (auto-download if missing) ----------------------------------
DATA_ZIP_ID = '1oT07-wVuYj_hByn4FqUtIj63Yg_fe7dP'

if not os.path.isdir('data/train'):
    print('data/ not found - downloading dataset (~52 MB) from Google Drive...')
    try:
        import gdown
        gdown.download(id=DATA_ZIP_ID, output='flood_dataset.zip', quiet=False)
        with zipfile.ZipFile('flood_dataset.zip') as z:
            z.extractall('.')
        if not os.path.isdir('data'):
            # zip may nest everything under a single root folder - find 'data' anywhere
            for root, dirs, _ in os.walk('.'):
                if os.path.basename(root) == 'data' and 'train' in dirs:
                    shutil.move(root, 'data')
                    break
    except Exception as e:
        raise RuntimeError(
            'Automatic download failed. Please download the dataset manually from '
            'https://drive.google.com/file/d/1oT07-wVuYj_hByn4FqUtIj63Yg_fe7dP/view '
            'and extract it so the folder layout is data/{train,valid,test}/{flooding,normal}.'
        ) from e

for split in ['train', 'valid', 'test']:
    n = sum(len(files) for _, _, files in os.walk(os.path.join('data', split)))
    print(f'{split:6s}: {n} images')

In [ ]:
# Build tf.data pipelines (modern replacement for the deprecated ImageDataGenerator)
def make_dataset(split, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        os.path.join('data', split),
        labels='inferred',
        label_mode='categorical',                # one-hot -> matches categorical_crossentropy
        class_names=['flooding', 'normal'],      # 0 = Flooding, 1 = No Flooding
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )
    # MobileNet expects inputs scaled to [-1, 1]
    return ds.map(lambda x, y: (tf.keras.applications.mobilenet.preprocess_input(x), y),
                  num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset('train', shuffle=True)
valid_ds = make_dataset('valid', shuffle=False)
test_ds  = make_dataset('test',  shuffle=False)   # shuffle=False keeps labels aligned

## 2️⃣ Build the fine-tuned model

We keep the convolutional base of MobileNet, attach a new classification head
(GlobalAveragePooling → Dropout → Dense(2, **softmax**)) and fine-tune only the top 23 layers.

> ℹ️ *Fix vs. the original notebook:* the head now uses **softmax** with
> `categorical_crossentropy`, so predicted scores form valid probabilities that sum to 1.

In [ ]:
# Construct the fine-tuned model ----------------------------------------------
def build_flood_model():
    mobile = tf.keras.applications.MobileNet(weights='imagenet', include_top=False)
    x = mobile.get_layer('conv_pw_13_relu').output      # last ReLU feature map (1024 ch)
    x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling2d')(x)
    x = tf.keras.layers.Dropout(0.2, name='dropout')(x)
    output = tf.keras.layers.Dense(units=2, activation='softmax', name='output')(x)
    return tf.keras.models.Model(inputs=mobile.input, outputs=output, name='flood_net')

model = build_flood_model()

In [ ]:
# Freeze the convolutional base - only the last 23 layers stay trainable --------
for layer in model.layers[:-23]:
    layer.trainable = False

trainable = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers: {trainable}/{len(model.layers)}')
model.summary()

In [ ]:
# Compile ----------------------------------------------------------------------
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

## 3️⃣ Train

Ten epochs with **EarlyStopping** (restores the best weights automatically), so training
stops as soon as the validation accuracy stops improving.

In [ ]:
# Train ------------------------------------------------------------------------
history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                                                restore_best_weights=True)],
    verbose=2,
)

In [ ]:
# Plot training curves ----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot(history.history['accuracy'], 'o-', label='train')
axes[0].plot(history.history['val_accuracy'], 's-', label='validation')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(history.history['loss'], 'o-', label='train')
axes[1].plot(history.history['val_loss'], 's-', label='validation')
axes[1].set_title('Loss'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=.3)
plt.show()

In [ ]:
# Save the fine-tuned model (modern .keras format) ------------------------------
os.makedirs('models', exist_ok=True)
model.save('models/fine_tuned_flood_detection_model.keras')
print('Saved to models/fine_tuned_flood_detection_model.keras')
# Reload anytime with:  model = tf.keras.models.load_model('models/fine_tuned_flood_detection_model.keras')

## 4️⃣ Evaluate on the test set

Accuracy, **precision, recall and F1-score** (recall matters most here — a missed flood is
a costly false negative), plus a confusion matrix.

In [ ]:
# Predict & score ----------------------------------------------------------------
y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_ds])
probs  = model.predict(test_ds, verbose=0)
y_pred = np.argmax(probs, axis=1)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro')
rec  = recall_score(y_true, y_pred, average='macro')
f1   = f1_score(y_true, y_pred, average='macro')

print(f'Accuracy : {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'F1 Score : {f1:.4f}')
print()
print(classification_report(y_true, y_pred, target_names=LABELS, digits=4))

In [ ]:
# Confusion matrix ----------------------------------------------------------------
def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    fig, ax = plt.subplots(figsize=(5.5, 4.5), constrained_layout=True)
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    fig.colorbar(im, ax=ax)
    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks, classes, rotation=45)
    ax.set_yticks(tick_marks, classes)
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        ax.text(j, i, format(cm[i, j], 'd'), ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')
    ax.set_ylabel('True label'); ax.set_xlabel('Predicted label'); ax.set_title(title)
    plt.show()

cm = confusion_matrix(y_true, y_pred)
plot_confusion_matrix(cm, classes=LABELS, title='Flood Detection - Confusion Matrix')

## 5️⃣ Run inference on your own images

Place any image in the `evaluate/` folder (or change `IMG_NAME` below) and get a verdict.

In [ ]:
# Prepare an image for the model ---------------------------------------------------
def preprocess_image(file, img_dir='evaluate/'):
    img = tf.keras.preprocessing.image.load_img(img_dir + file, target_size=IMG_SIZE)
    arr = tf.keras.preprocessing.image.img_to_array(img)
    return tf.keras.applications.mobilenet.preprocess_input(np.expand_dims(arr, axis=0))

IMG_NAME = '1.jpg'          # <-- change me (any file inside evaluate/)
from IPython.display import Image as IPyImage, display
display(IPyImage(filename='evaluate/' + IMG_NAME, width=320))

In [ ]:
# Predict ----------------------------------------------------------------------------
pred = model.predict(preprocess_image(IMG_NAME), verbose=0)[0]
result = int(np.argmax(pred))
print(f'File                : {IMG_NAME}')
print(f'Flooding  score     : {pred[0]:.4f}')
print(f'No Flooding score   : {pred[1]:.4f}')
print(f'VERDICT             : {LABELS[result]}  (confidence {pred[result]*100:.1f}%)')

---
### Credits

* Code inspired by and adapted from [deeplizard](https://deeplizard.com/).
* "Flooding" images courtesy of the authors of [*Detecting floodwater on roadways from image
  data with handcrafted features and deep transfer learning*](https://arxiv.org/pdf/1909.00125.pdf).
* "No Flooding" images collected via automated Google image search (some label noise expected).